# CodingSSM Training — Kaggle GPU

Run SFT + GRPO on a free Kaggle T4 GPU.

## Setup
1. Upload your training data to a Kaggle Dataset (see cell 2)
2. Enable GPU: Settings → Accelerator → GPU T4 x2
3. Run all cells — resumes automatically from last checkpoint

**Expected runtime**: SFT ~45 min, GRPO ~6-8 hours (fits in one 9h session)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mamba-ssm', 'causal-conv1d', 'safetensors', 'transformers',
    'datasets', 'huggingface_hub', 'faiss-cpu', 'sentence-transformers'
], check=True)
print('Dependencies installed')

In [ ]:
# ── Cell 2: Pull repo + data from HuggingFace ─────────────────────────────────
# Option A: from HuggingFace Hub (upload your data there first)
# Option B: clone your GitHub repo

import os
from pathlib import Path

REPO = 'your-username/CodingSSM'          # ← change this
HF_TOKEN = os.environ.get('HF_TOKEN', '') # set in Kaggle Secrets

# Clone repo
if not Path('CodingSSM').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', 'CodingSSM'], check=True)

os.chdir('CodingSSM')
sys.path.insert(0, '.')

# Pull training traces from HuggingFace dataset
from huggingface_hub import hf_hub_download, snapshot_download

DATA_REPO = 'your-username/codingssm-traces'  # ← your HF dataset repo
Path('data').mkdir(exist_ok=True)

if HF_TOKEN and DATA_REPO:
    for fname in ['all_traces_v2.jsonl', 'reasoning_traces_r1.jsonl', 'epichat_traces.jsonl']:
        try:
            hf_hub_download(
                repo_id=DATA_REPO, filename=fname,
                repo_type='dataset', token=HF_TOKEN,
                local_dir='data'
            )
            print(f'Downloaded {fname}')
        except Exception as e:
            print(f'Could not download {fname}: {e}')

# Check what data we have
for p in Path('data').glob('*.jsonl'):
    lines = sum(1 for _ in open(p))
    print(f'  {p.name}: {lines} traces')

In [ ]:
# ── Cell 3: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

In [ ]:
# ── Cell 4: SFT Training ──────────────────────────────────────────────────────
# Skip if checkpoint already exists
import subprocess
from pathlib import Path

SFT_CKPT = Path('checkpoints/sft_v2/sft_best.pt')

if SFT_CKPT.exists():
    print(f'SFT checkpoint exists: {SFT_CKPT} — skipping')
else:
    # Pick best available traces
    traces = 'data/all_traces_v2.jsonl'
    if not Path(traces).exists():
        traces = 'data/epichat_traces.jsonl'
    n = sum(1 for _ in open(traces))
    print(f'Training SFT on {n} traces from {traces}...')

    subprocess.run([
        sys.executable, '-m', 'train.sft_reasoning',
        '--traces', traces,
        '--output-dir', 'checkpoints/sft_v2',
        '--model-size', '700m',
        '--epochs', '3',
        '--lr', '2e-4',
        '--batch-size', '4',      # larger batch on GPU
        '--grad-accum', '4',
        '--max-seq-len', '1024',
        '--device', DEVICE,
    ], check=True)

SFT_CKPT = sorted(Path('checkpoints/sft_v2').glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
print(f'SFT done: {SFT_CKPT}')

In [ ]:
# ── Cell 5: GRPO Training ─────────────────────────────────────────────────────
from pathlib import Path

GRPO_DIR = Path('checkpoints/grpo')
GRPO_DIR.mkdir(exist_ok=True)

existing = sorted(GRPO_DIR.glob('*.pt'), key=lambda p: p.stat().st_mtime)
if existing:
    resume = existing[-1]
    print(f'Resuming GRPO from: {resume}')
else:
    resume = SFT_CKPT
    print(f'Starting GRPO from SFT: {resume}')

traces = 'data/all_traces_v2.jsonl'
if not Path(traces).exists():
    traces = 'data/epichat_traces.jsonl'

subprocess.run([
    sys.executable, '-m', 'train.grpo',
    '--traces', traces,
    '--checkpoint', str(resume),
    '--output-dir', str(GRPO_DIR),
    '--model-size', '700m',
    '--group-size', '8',
    '--lr', '5e-6',
    '--max-steps', '3000',
    '--kl-coeff', '0.04',
    '--max-new-tokens', '1024',
    '--temperature', '0.8',
    '--device', DEVICE,
], check=True)

GRPO_CKPT = sorted(GRPO_DIR.glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
print(f'GRPO done: {GRPO_CKPT}')

In [ ]:
# ── Cell 6: Export + Upload to HuggingFace ────────────────────────────────────
from pathlib import Path
import os

HF_REPO = 'your-username/CodingSSM-1.6B'  # ← change this
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# Find best checkpoint
candidates = [
    *sorted(Path('checkpoints/self_improve').rglob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True),
    *sorted(Path('checkpoints/grpo').glob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True),
    *sorted(Path('checkpoints/sft_v2').glob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True),
]
best = candidates[0]
print(f'Exporting: {best}')

subprocess.run([
    sys.executable, 'scripts/export_model.py',
    '--checkpoint', str(best),
    '--version', '0.1.0',
], check=True)

if HF_TOKEN and HF_REPO:
    subprocess.run([
        sys.executable, 'scripts/export_model.py',
        '--checkpoint', str(best),
        '--version', '0.1.0',
        '--push', '--repo', HF_REPO,
    ], env={**os.environ, 'HF_TOKEN': HF_TOKEN}, check=True)
    print(f'Uploaded to https://huggingface.co/{HF_REPO}')
else:
    print('Set HF_TOKEN in Kaggle Secrets to auto-upload')
    print('Export saved to dist/')

In [ ]:
# ── Cell 7: Quick eval on HumanEval (optional) ────────────────────────────────
# Runs pass@1 on first 20 problems to get a quick benchmark score
subprocess.run([
    sys.executable, 'scripts/eval_humaneval.py',
    '--n-problems', '20',
    '--n-samples', '4',
    '--checkpoint', str(best),
], check=True)